# DPO data preparation — shared 599-row reference cache

This version is optimized for the case where every country has the **same normalized prompts and the same two response texts**, while `chosen`/`rejected` may differ by country.

It:

1. reads files named `D_syn_COUNTRYNAME_599.jsonl` (for example, `D_syn_USA_599.jsonl`);
2. normalizes prompts by adding/fixing the `Context:` prefix;
3. renames GPS dimensions exactly as in `preprocessing_without_gpu_multi_country.ipynb`;
4. verifies that countries share the same questions and the same unordered response pair for every question;
5. uses one shared train/eval split for all countries;
6. **reuses reference log-probabilities if a reusable ref file already exists**;
7. otherwise computes reference log-probabilities **once per prompt/response pair** on GPU and saves a shared cache; and
8. writes country-specific train, eval, and `train_with_ref` files using each country's own `chosen`/`rejected` labels.

If a previous country-specific `*_train_with_ref.jsonl` file already exists, the notebook can reconstruct the shared cache from it, so the expensive base-model pass does not have to be repeated.


In [1]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.6 MB/s eta 0:00:00


In [ ]:
# Hugging Face login is performed later only if no reusable reference file exists.
# This lets the notebook rebuild country files from an existing ref cache without
# requiring a GPU or a Hugging Face login.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
from pathlib import Path

# Use the short tag used by the training notebook / adapter folder.
# US maps to the raw/data filename code USA.
COUNTRY_CODES = ["CHN", "JPN", "GBR", "US", "MEX", "ARG", "DEU"]

COUNTRY_NAME_MAP = {
    "US": "USA",
}


def resolve_country_codes(country_code):
    data_country_code = COUNTRY_NAME_MAP.get(country_code, country_code)
    adapter_tag = country_code
    return data_country_code, adapter_tag


# Raw files: D_syn_USA_599.jsonl, D_syn_MEX_599.jsonl, etc.
RAW_DATA_DIR = Path("/content/drive/MyDrive/DPO/pre-processing")

# Prepared files and the reusable reference cache are stored here.
DATA_DIR = Path("/content/drive/MyDrive/DPO")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Preferred reusable cache. It contains one row per (prompt, completion) pair.
SHARED_REF_FILE = DATA_DIR / "shared_reference_logps.jsonl"
REF_MANIFEST_FILE = DATA_DIR / "shared_reference_logps_manifest.json"


def build_country_paths(country_code):
    data_country_code, adapter_tag = resolve_country_codes(country_code)

    raw_file = RAW_DATA_DIR / f"D_syn_{data_country_code}.jsonl"
    train_file = DATA_DIR / f"{data_country_code}_train.jsonl"
    eval_file = DATA_DIR / f"{data_country_code}_eval.jsonl"
    out_file = DATA_DIR / f"{data_country_code}_train_with_ref.jsonl"

    return raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag


for country_code in COUNTRY_CODES:
    print(country_code, "->", build_country_paths(country_code))
print("Shared ref cache ->", SHARED_REF_FILE)


CHN -> (PosixPath('/content/drive/MyDrive/DPO/pre-processing/D_syn_CHN.jsonl'), PosixPath('/content/drive/MyDrive/DPO/CHN_train.jsonl'), PosixPath('/content/drive/MyDrive/DPO/CHN_eval.jsonl'), PosixPath('/content/drive/MyDrive/DPO/CHN_train_with_ref.jsonl'), 'CHN', 'CHN')
JPN -> (PosixPath('/content/drive/MyDrive/DPO/pre-processing/D_syn_JPN.jsonl'), PosixPath('/content/drive/MyDrive/DPO/JPN_train.jsonl'), PosixPath('/content/drive/MyDrive/DPO/JPN_eval.jsonl'), PosixPath('/content/drive/MyDrive/DPO/JPN_train_with_ref.jsonl'), 'JPN', 'JPN')
GBR -> (PosixPath('/content/drive/MyDrive/DPO/pre-processing/D_syn_GBR.jsonl'), PosixPath('/content/drive/MyDrive/DPO/GBR_train.jsonl'), PosixPath('/content/drive/MyDrive/DPO/GBR_eval.jsonl'), PosixPath('/content/drive/MyDrive/DPO/GBR_train_with_ref.jsonl'), 'GBR', 'GBR')
US -> (PosixPath('/content/drive/MyDrive/DPO/pre-processing/D_syn_USA.jsonl'), PosixPath('/content/drive/MyDrive/DPO/USA_train.jsonl'), PosixPath('/content/drive/MyDrive/DPO/USA_eva

In [4]:
import os
import json
import random

# These settings matter only if the GPU reference pass is needed.
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_BF16"] = "1"

import torch

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
TRAIN_FRAC = 0.80
SEED = 42
MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 256
PROMPT_FORMAT_VERSION = "questionnaire_situation_v1"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. That is fine if an existing ref cache/ref file can be reused.")


CUDA available: True
CUDA: 12.8
GPU: Tesla T4


In [5]:
# Same renames as preprocessing_without_gpu_multi_country.ipynb.
DIMENSION_RENAMES = {
    "negrecip": "negative_reciprocity",
    "posrecip": "positive_reciprocity",
    "risktaking": "risk_taking",
}

REQUIRED_DPO_FIELDS = ("prompt", "chosen", "rejected")


def normalize_prompt(text):
    if not isinstance(text, str) or not text.strip():
        return text

    text = text.strip()
    first = text.split()[0]

    if first == "Context:":
        return text
    if first == "context:":
        return text[:1].upper() + text[1:]
    return "Context: " + text


def normalize_row(row):
    row = dict(row)
    row["prompt"] = normalize_prompt(row.get("prompt"))

    for field in ("chosen", "rejected"):
        value = row.get(field)
        if isinstance(value, str):
            row[field] = value.strip()

    if "gps_dimension" in row:
        row["gps_dimension"] = DIMENSION_RENAMES.get(
            row["gps_dimension"], row["gps_dimension"]
        )
    return row


def validate_rows(rows, source_name="dataset"):
    if not rows:
        raise ValueError(f"{source_name} is empty")

    for i, row in enumerate(rows):
        for field in REQUIRED_DPO_FIELDS:
            value = row.get(field)
            if not isinstance(value, str) or not value.strip():
                raise ValueError(
                    f"{source_name}: row {i} has missing/empty required field {field!r}"
                )
        if row["chosen"] == row["rejected"]:
            raise ValueError(f"{source_name}: row {i} has identical chosen and rejected")


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_country(raw_path, country_name):
    rows = [normalize_row(row) for row in load_jsonl(raw_path)]
    validate_rows(rows, source_name=str(raw_path))

    by_prompt = {}
    for row in rows:
        p = row["prompt"]
        if p in by_prompt:
            raise ValueError(
                f"{raw_path}: duplicate normalized prompt found. "
                "This shared-cache version expects one DPO row per prompt."
            )
        by_prompt[p] = row

    print(f"{country_name}: loaded {len(rows)} rows / {len(by_prompt)} unique prompts")
    return rows, by_prompt


def response_pair(row):
    return frozenset((row["chosen"], row["rejected"]))


def verify_shared_questions(country_rows_by_prompt):
    reference_code = COUNTRY_CODES[0]
    reference_map = country_rows_by_prompt[reference_code]
    reference_prompts = set(reference_map)

    for country_code in COUNTRY_CODES[1:]:
        current_map = country_rows_by_prompt[country_code]
        current_prompts = set(current_map)

        missing = reference_prompts - current_prompts
        extra = current_prompts - reference_prompts
        if missing or extra:
            raise ValueError(
                f"{country_code}: prompt set differs from {reference_code}. "
                f"Missing={len(missing)}, extra={len(extra)}."
            )

        for prompt in reference_prompts:
            if response_pair(reference_map[prompt]) != response_pair(current_map[prompt]):
                raise ValueError(
                    f"{country_code}: response texts differ for prompt:\n{prompt[:250]}...\n"
                    "The shared ref cache is valid only when the two response texts are "
                    "the same across countries (their chosen/rejected order may differ)."
                )

    print(
        f"Verified {len(reference_prompts)} shared prompts across {len(COUNTRY_CODES)} countries; "
        "each prompt has the same two response texts in every country."
    )
    return reference_prompts


def make_shared_split(all_prompts, train_frac=TRAIN_FRAC, seed=SEED):
    prompt_list = sorted(all_prompts)
    rng = random.Random(seed)
    rng.shuffle(prompt_list)
    n_train = int(len(prompt_list) * train_frac)
    return set(prompt_list[:n_train]), set(prompt_list[n_train:])


def cache_from_full_ref_rows(rows, source_name):
    cache = {}
    prompts = set()

    for i, raw_row in enumerate(rows):
        row = normalize_row(raw_row)
        p = row.get("prompt")
        chosen = row.get("chosen")
        rejected = row.get("rejected")
        rc = row.get("ref_chosen_logps")
        rr = row.get("ref_rejected_logps")

        if not p or not chosen or not rejected or rc is None or rr is None:
            raise ValueError(
                f"{source_name}: row {i} is not a usable train_with_ref row."
            )

        prompts.add(p)
        for completion, value in ((chosen, rc), (rejected, rr)):
            key = (p, completion)
            value = float(value)
            if key in cache and abs(cache[key] - value) > 1e-5:
                raise ValueError(f"{source_name}: conflicting ref logp for one prompt/completion")
            cache[key] = value

    return cache, prompts


def load_shared_cache(path):
    rows = load_jsonl(path)
    if not rows:
        raise ValueError(f"Shared ref cache is empty: {path}")

    # Preferred compact cache schema.
    if {"prompt", "completion", "ref_logp"}.issubset(rows[0]):
        cache = {}
        prompts = set()
        for i, row in enumerate(rows):
            p = normalize_prompt(row.get("prompt"))
            completion = row.get("completion")
            value = row.get("ref_logp")
            if not p or not isinstance(completion, str) or value is None:
                raise ValueError(f"{path}: invalid cache row {i}")
            completion = completion.strip()
            cache[(p, completion)] = float(value)
            prompts.add(p)
        return cache, prompts

    # Also accept a country-specific *_train_with_ref.jsonl directly.
    return cache_from_full_ref_rows(rows, str(path))


def write_shared_cache(cache, path):
    rows = [
        {"prompt": prompt, "completion": completion, "ref_logp": value}
        for (prompt, completion), value in sorted(cache.items())
    ]
    write_jsonl(rows, path)


def validate_cache_coverage(cache, train_prompts, shared_response_pairs, expected_train_count):
    if len(train_prompts) != expected_train_count:
        raise ValueError(
            f"Existing ref file implies {len(train_prompts)} training prompts, but "
            f"TRAIN_FRAC={TRAIN_FRAC} with the current data expects {expected_train_count}. "
            "Delete/rename the stale ref file or restore the matching split settings."
        )

    missing = []
    for prompt in sorted(train_prompts):
        for completion in shared_response_pairs[prompt]:
            if (prompt, completion) not in cache:
                missing.append((prompt, completion))
                if len(missing) >= 5:
                    break
        if len(missing) >= 5:
            break

    if missing:
        examples = [p[:80] for p, _ in missing]
        raise ValueError(
            "Existing ref file does not cover both response texts for all training prompts. "
            f"Examples: {examples}"
        )


def find_existing_country_ref_file():
    # Prefer new 599 filenames, but also accept the immediately previous no-suffix version.
    candidates = []
    for country_code in COUNTRY_CODES:
        _, _, _, new_ref, data_country_code, _ = build_country_paths(country_code)
        candidates.append(new_ref)
        candidates.append(DATA_DIR / f"{data_country_code}_train_with_ref.jsonl")

    for path in candidates:
        if path.exists():
            return path
    return None


In [8]:
# Load and normalize all raw country files first.
country_rows = {}
country_rows_by_prompt = {}

for country_code in COUNTRY_CODES:
    raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag = build_country_paths(country_code)

    if not raw_file.exists():
        raise FileNotFoundError(
            f"Raw file not found: {raw_file}. Expected D_syn_{data_country_code}.jsonl"
        )

    rows, by_prompt = load_country(raw_file, data_country_code)
    country_rows[country_code] = rows
    country_rows_by_prompt[country_code] = by_prompt

all_prompts = verify_shared_questions(country_rows_by_prompt)
expected_train_count = int(len(all_prompts) * TRAIN_FRAC)

# The unordered pair of response texts is common across countries.
reference_map = country_rows_by_prompt[COUNTRY_CODES[0]]
shared_response_pairs = {
    prompt: response_pair(reference_map[prompt])
    for prompt in all_prompts
}

print(f"Expected shared split: {expected_train_count} train / {len(all_prompts) - expected_train_count} eval prompts")


CHN: loaded 658 rows / 658 unique prompts
JPN: loaded 658 rows / 658 unique prompts
GBR: loaded 658 rows / 658 unique prompts
USA: loaded 658 rows / 658 unique prompts
MEX: loaded 658 rows / 658 unique prompts
ARG: loaded 658 rows / 658 unique prompts
DEU: loaded 658 rows / 658 unique prompts
Verified 658 shared prompts across 7 countries; each prompt has the same two response texts in every country.
Expected shared split: 526 train / 132 eval prompts


In [9]:
def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def compute_shared_ref_cache_on_gpu(train_prompts, shared_response_pairs):
    """Compute each shared (prompt, response) log-probability exactly once."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No reusable ref file was found and CUDA is unavailable. "
            "Run this branch on a GPU once, or provide an existing shared/country ref file."
        )

    # Login only when weights actually need to be loaded.
    from huggingface_hub import notebook_login
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    notebook_login()

    compute_dtype = torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading reference (base) model in 4-bit...")
    ref_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )
    ref_model.eval()

    def format_prompt_text(prompt_text: str) -> str:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": build_user_prompt(prompt_text)}],
            tokenize=False,
            add_generation_prompt=True,
        )

    @torch.no_grad()
    def seq_logprob_for_completion(prompt_text: str, completion_text: str) -> float:
        prompt = format_prompt_text(prompt_text)

        prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
        if len(prompt_ids) > MAX_PROMPT_TOKENS:
            prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]

        comp_ids = tokenizer(completion_text, add_special_tokens=False).input_ids
        if len(comp_ids) > MAX_COMPLETION_TOKENS:
            comp_ids = comp_ids[:MAX_COMPLETION_TOKENS]

        input_ids = prompt_ids + comp_ids
        if len(input_ids) < 2 or not comp_ids:
            return float("-inf")

        labels = ([-100] * len(prompt_ids)) + comp_ids
        input_ids_t = torch.tensor([input_ids], device=ref_model.device)
        labels_t = torch.tensor([labels], device=ref_model.device)

        logits = ref_model(input_ids=input_ids_t).logits
        log_probs = torch.log_softmax(logits, dim=-1)[:, :-1, :]
        labels_shifted = labels_t[:, 1:]

        mask = labels_shifted.ne(-100)
        gold = labels_shifted.clone()
        gold[~mask] = 0
        token_logps = log_probs.gather(-1, gold.unsqueeze(-1)).squeeze(-1)
        token_logps = token_logps * mask
        return float(token_logps.sum().detach().cpu().to(torch.float32))

    cache = {}
    ordered_prompts = sorted(train_prompts)
    for i, prompt in enumerate(ordered_prompts, 1):
        for completion in sorted(shared_response_pairs[prompt]):
            cache[(prompt, completion)] = seq_logprob_for_completion(prompt, completion)

        if i % 25 == 0 or i == len(ordered_prompts):
            print(f"  scored {i}/{len(ordered_prompts)} prompts "
                  f"({2 * i} prompt/response pairs)")

    del ref_model
    torch.cuda.empty_cache()
    return cache


## Reuse or compute the shared reference cache

Priority:

1. `shared_599_reference_logps.jsonl` if it already exists;
2. otherwise the first existing country-specific `*_599_train_with_ref.jsonl` (or legacy `*_train_with_ref.jsonl`), from which the cache is reconstructed;
3. otherwise one GPU pass over the shared training prompts and their two response texts.

When an existing country ref file is reused, its training prompt set becomes the shared split so all countries are rebuilt consistently with that old ref file.


In [11]:
# ---------- 1) Find/reuse a reference source, or define a new shared split ----------
ref_cache = None
ref_source = None

if SHARED_REF_FILE.exists():
    print("Found shared ref cache:", SHARED_REF_FILE)
    ref_cache, train_prompts = load_shared_cache(SHARED_REF_FILE)
    ref_source = str(SHARED_REF_FILE)
else:
    old_ref_file = find_existing_country_ref_file()
    if old_ref_file is not None:
        print("Shared cache not found; reusing existing country ref file:", old_ref_file)
        ref_cache, train_prompts = load_shared_cache(old_ref_file)
        ref_source = str(old_ref_file)
    else:
        print("No reusable ref file found. A single GPU reference pass will be run.")
        train_prompts, _ = make_shared_split(all_prompts)

# The existing ref file defines the split when one is reused.
train_prompts = set(train_prompts)
if not train_prompts.issubset(all_prompts):
    unknown = train_prompts - all_prompts
    raise ValueError(
        f"Reference file contains {len(unknown)} prompts that are absent from the current raw data."
    )

eval_prompts = set(all_prompts) - train_prompts

if ref_cache is not None:
    validate_cache_coverage(
        ref_cache,
        train_prompts,
        shared_response_pairs,
        expected_train_count,
    )

print(f"Using shared split: {len(train_prompts)} train / {len(eval_prompts)} eval prompts")


# ---------- 2) Create aligned train/eval files for every country ----------
prepared_splits = {}

for country_code in COUNTRY_CODES:
    raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag = build_country_paths(country_code)
    by_prompt = country_rows_by_prompt[country_code]

    train_rows = []
    for i, prompt in enumerate(sorted(train_prompts)):
        row = dict(by_prompt[prompt])
        row["country"] = data_country_code
        row["item_id"] = f"{data_country_code}_train_{i:04d}"
        train_rows.append(row)

    eval_rows = []
    for i, prompt in enumerate(sorted(eval_prompts)):
        row = dict(by_prompt[prompt])
        row["country"] = data_country_code
        row["item_id"] = f"{data_country_code}_eval_{i:04d}"
        eval_rows.append(row)

    write_jsonl(train_rows, train_file)
    write_jsonl(eval_rows, eval_file)
    prepared_splits[country_code] = (train_rows, eval_rows, out_file)

    print(f"{data_country_code}: train {len(train_rows)} -> {train_file}")
    print(f"{data_country_code}: eval  {len(eval_rows)} -> {eval_file}")


# ---------- 3) If necessary, compute shared log-probs ONCE on GPU ----------
if ref_cache is None:
    ref_cache = compute_shared_ref_cache_on_gpu(train_prompts, shared_response_pairs)
    validate_cache_coverage(
        ref_cache,
        train_prompts,
        shared_response_pairs,
        expected_train_count,
    )
    write_shared_cache(ref_cache, SHARED_REF_FILE)
    ref_source = str(SHARED_REF_FILE)
    print("Saved new shared ref cache:", SHARED_REF_FILE)
else:
    # If we reconstructed from an old country file, save the compact shared cache
    # so future runs can use it directly.
    if not SHARED_REF_FILE.exists():
        write_shared_cache(ref_cache, SHARED_REF_FILE)
        print("Created compact shared ref cache from existing ref file:", SHARED_REF_FILE)


# ---------- 4) Create each country's train_with_ref using its own preference direction ----------
for country_code in COUNTRY_CODES:
    train_rows, _, out_file = prepared_splits[country_code]

    rows_with_ref = []
    for ex in train_rows:
        row = dict(ex)
        row["ref_chosen_logps"] = ref_cache[(row["prompt"], row["chosen"])]
        row["ref_rejected_logps"] = ref_cache[(row["prompt"], row["rejected"])]
        rows_with_ref.append(row)

    write_jsonl(rows_with_ref, out_file)
    print(f"{country_code}: wrote {len(rows_with_ref)} train_with_ref rows -> {out_file}")


# ---------- 5) Write a small manifest describing the cache/split ----------
manifest = {
    "model_name": MODEL_NAME,
    "prompt_format_version": PROMPT_FORMAT_VERSION,
    "max_prompt_tokens": MAX_PROMPT_TOKENS,
    "max_completion_tokens": MAX_COMPLETION_TOKENS,
    "train_frac": TRAIN_FRAC,
    "seed": SEED,
    "n_total_prompts": len(all_prompts),
    "n_train_prompts": len(train_prompts),
    "n_eval_prompts": len(eval_prompts),
    "reference_source": ref_source,
    "shared_ref_file": str(SHARED_REF_FILE),
    "countries": COUNTRY_CODES,
}
with open(REF_MANIFEST_FILE, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Wrote manifest:", REF_MANIFEST_FILE)
print("\nDone. All countries now share the same train/eval questions and the same reusable ref cache.")


No reusable ref file found. A single GPU reference pass will be run.
Using shared split: 526 train / 132 eval prompts
CHN: train 526 -> /content/drive/MyDrive/DPO/CHN_train.jsonl
CHN: eval  132 -> /content/drive/MyDrive/DPO/CHN_eval.jsonl
JPN: train 526 -> /content/drive/MyDrive/DPO/JPN_train.jsonl
JPN: eval  132 -> /content/drive/MyDrive/DPO/JPN_eval.jsonl
GBR: train 526 -> /content/drive/MyDrive/DPO/GBR_train.jsonl
GBR: eval  132 -> /content/drive/MyDrive/DPO/GBR_eval.jsonl
USA: train 526 -> /content/drive/MyDrive/DPO/USA_train.jsonl
USA: eval  132 -> /content/drive/MyDrive/DPO/USA_eval.jsonl
MEX: train 526 -> /content/drive/MyDrive/DPO/MEX_train.jsonl
MEX: eval  132 -> /content/drive/MyDrive/DPO/MEX_eval.jsonl
ARG: train 526 -> /content/drive/MyDrive/DPO/ARG_train.jsonl
ARG: eval  132 -> /content/drive/MyDrive/DPO/ARG_eval.jsonl
DEU: train 526 -> /content/drive/MyDrive/DPO/DEU_train.jsonl
DEU: eval  132 -> /content/drive/MyDrive/DPO/DEU_eval.jsonl


Loading tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

  scored 25/526 prompts (50 prompt/response pairs)
  scored 50/526 prompts (100 prompt/response pairs)
  scored 75/526 prompts (150 prompt/response pairs)
  scored 100/526 prompts (200 prompt/response pairs)
  scored 125/526 prompts (250 prompt/response pairs)
  scored 150/526 prompts (300 prompt/response pairs)
  scored 175/526 prompts (350 prompt/response pairs)
  scored 200/526 prompts (400 prompt/response pairs)
  scored 225/526 prompts (450 prompt/response pairs)
  scored 250/526 prompts (500 prompt/response pairs)
  scored 275/526 prompts (550 prompt/response pairs)
  scored 300/526 prompts (600 prompt/response pairs)
  scored 325/526 prompts (650 prompt/response pairs)
  scored 350/526 prompts (700 prompt/response pairs)
  scored 375/526 prompts (750 prompt/response pairs)
  scored 400/526 prompts (800 prompt/response pairs)
  scored 425/526 prompts (850 prompt/response pairs)
  scored 450/526 prompts (900 prompt/response pairs)
  scored 475/526 prompts (950 prompt/response pair